# Decision Trees — AI Lab Instructor Notebook

*Classical ML · Medium*

Understand decision trees deeply (Gini/entropy, split selection, overfitting, pruning) and be able to explain and debug tree behavior like a FAANG ML engineer.

**Outcome.** Students can:
- compute impurity and information gain,
- implement a 1-level decision stump from scratch,
- train a sklearn DecisionTree as a sanity check,
- diagnose overfitting and data leakage risks.

8 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Decision Trees — Student Lab

We start using **sklearn** in Week 4, but you’ll still implement core pieces from scratch.

## Section 0 — Synthetic dataset
We’ll create a non-linear boundary dataset to show how trees fit.

### Figure: Synthetic Nonlinear Dataset

![Synthetic Nonlinear Dataset](/images/w4-d1-synthetic-nonlinear.png)

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

def make_nonlinear(n=400):
    X = rng.uniform(-2, 2, size=(n, 2))
    r = np.sqrt(X[:,0]**2 + X[:,1]**2)
    y = (r < 1.0).astype(int)
    flip = rng.random(n) < 0.05
    # we take the original label and switch it: if it was 0, it becomes 1, and if it was 1, it becomes 0
    y[flip] = 1 - y[flip]
    return X, y

X, y = make_nonlinear()
n = X.shape[0]
idx = rng.permutation(n)
tr = idx[: int(0.7*n)]
va = idx[int(0.7*n):]
Xtr, ytr = X[tr], y[tr]
Xva, yva = X[va], y[va]
check('shapes', Xtr.shape[0]==ytr.shape[0] and Xva.shape[0]==yva.shape[0])

## Task 2: Gini impurity
*Section: Impurity*

## Section 1 — Impurity

### Task 1.1: Gini impurity

![DtreeMiniEx.png](/images/w4-d1-decision-tree-1-5f3ede58.png)
![Gini Impurity Split Comparison](/images/w4-d1-gini-impurity-split-comparison.png)

**Formula:** `Gini(S) = 1 - sum_k p_k^2`

Binary case: `Gini(S) = 1 - (p0^2 + p1^2)`

**FAANG gotcha:** Gini is usually faster than entropy in training because it avoids logarithm computations.

**Hints**
- p_k = count_k / n; gini = 1 - sum(p_k^2)
- p_k = count_k / n; gini = 1 - sum(p_k^2)

In [ ]:
def gini(y):
    y = np.asarray(y, dtype=int)
    if y.size == 0:
        return 0.0
    p1 = y.mean()
    p0 = 1 - p1
    return float(1 - (p0*p0 + p1*p1))

check('gini_half', abs(gini(np.array([0,1]*5)) - 0.5) < 1e-12)

# Rationale: impurity measures label mix; pure node -> 0.


In [ ]:
# Checks
check('gini_half', abs(gini(np.array([0,1]*5)) - 0.5) < 1e-12)

**Why this works.** impurity measures label mix; pure node -> 0.

## Task 3: Entropy
*Section: Impurity*

### Task 1.2: Entropy

**Formula:** `Entropy(S) = -sum_k p_k log2(p_k)`

Binary case: `Entropy(S) = -(p0 log2(p0) + p1 log2(p1))`

**Hints**
- entropy = -sum p log2 p (use eps)

In [ ]:
def entropy(y):
    y = np.asarray(y, dtype=int)
    if y.size == 0:
        return 0.0
    p1 = y.mean()
    p0 = 1 - p1
    if p0 == 0.0 or p1 == 0.0:
        return 0.0
    return float(-(p0*np.log2(p0) + p1*np.log2(p1)))

check('entropy_half', abs(entropy(np.array([0,1]*5)) - 1.0) < 1e-9)

# Rationale: entropy measures uncertainty; pure node -> 0, max at 50/50 split.


In [ ]:
# Checks
check('entropy_half', abs(entropy(np.array([0,1]*5)) - 1.0) < 1e-9)

**Why this works.** entropy measures uncertainty; pure node -> 0, max at 50/50 split.

## Task 4: Evaluate impurity after threshold split
*Section: Best split (decision stump)*

## Section 2 — Best split (decision stump)

### Task 2.1: Evaluate impurity after threshold split

Split rule: go left if X[:,j] <= t else right.
Return weighted impurity and information gain.

In [ ]:
def info_gain(y, y_left, y_right, criterion='gini'):
    f = gini if criterion=='gini' else entropy
    parent = f(y)
    n = y.size
    wl = (y_left.size / n) * f(y_left)
    wr = (y_right.size / n) * f(y_right)
    return float(parent - (wl + wr))

# Evaluate impurity/gain for a simple threshold split
j = 0
t = float(np.median(Xtr[:, j]))
left = Xtr[:, j] <= t
right = ~left
ig = info_gain(ytr, ytr[left], ytr[right], criterion='gini')
print('info_gain', ig)
check('ig_nonneg', ig >= 0)

# Rationale: information gain is parent impurity minus weighted child impurity.


In [ ]:
# Checks
check('ig_nonneg', ig >= 0)

**Why this works.** information gain is parent impurity minus weighted child impurity.

## Task 5: Find best (feature, threshold)
*Section: Best split (decision stump)*

### Task 2.2: Find best (feature, threshold)


**FAANG gotcha:** if a split makes an empty child, skip it.

**Hints**
- thresholds from sorted unique feature values midpoints

In [ ]:
def best_split(X, y, criterion='gini'):
    best = (-1, None, -1.0)
    n, d = X.shape
    for j in range(d):
        vals = np.unique(X[:, j])
        if vals.size < 2:
            continue
        thresholds = (vals[:-1] + vals[1:]) / 2
        for t in thresholds:
            left = X[:, j] <= t
            right = ~left
            if left.sum() == 0 or right.sum() == 0:
                continue
            gain = info_gain(y, y[left], y[right], criterion=criterion)
            if gain > best[2]:
                best = (j, float(t), gain)
    return best

j, t, gain = best_split(Xtr, ytr)
print('best split', j, t, gain)
check('gain_nonneg', gain >= 0)

# Rationale: thresholds as midpoints avoid empty/duplicate splits.

In [ ]:
# Checks
check('gain_nonneg', gain >= 0)

**Why this works.** thresholds as midpoints avoid empty/duplicate splits.

## Task 6: Train a stump and evaluate
*Section: Best split (decision stump)*

### Task 2.3: Train a stump and evaluate

Use best_split to build a stump that predicts majority class on each side.

In [ ]:
def stump_predict(X_train, y_train, X_test, criterion='gini'):
    j, t, _ = best_split(X_train, y_train, criterion=criterion)
    left = X_train[:, j] <= t
    right = ~left

    left_label = int(np.round(y_train[left].mean()))
    right_label = int(np.round(y_train[right].mean()))

    test_left = X_test[:, j] <= t
    yhat = np.empty(X_test.shape[0], dtype=int)
    yhat[test_left] = left_label
    yhat[~test_left] = right_label
    return yhat

def accuracy(y, yhat):
    return float(np.mean(y == yhat))

yhat_tr = stump_predict(Xtr, ytr, Xtr)
yhat_va = stump_predict(Xtr, ytr, Xva)
print('stump train', accuracy(ytr, yhat_tr), 'val', accuracy(yva, yhat_va))

# Rationale: stump is weak learner; full tree captures nonlinear boundary.

**Why this works.** stump is weak learner; full tree captures nonlinear boundary.

## Task 7: Train trees with different max_depth
*Section: sklearn DecisionTreeClassifier (sanity check)*

## Section 3 — sklearn DecisionTreeClassifier (sanity check)

### Task 3.1: Train trees with different max_depth

In [ ]:
depths = [1,2,3,5,6,7,None]
for md in depths:
    clf = DecisionTreeClassifier(max_depth=md, random_state=0)
    clf.fit(Xtr, ytr)
    print('max_depth', md, 'train', clf.score(Xtr, ytr), 'val', clf.score(Xva, yva))

# Rationale: deeper trees reduce bias but increase variance; control with max_depth/min_samples_leaf.

**Why this works.** deeper trees reduce bias but increase variance; control with max_depth/min_samples_leaf.

## Task 8: Create a leaky feature
*Section: Failure mode: leakage*

## Section 4 — Failure mode: leakage

### Task 4.1: Create a leaky feature
Add a feature that is directly derived from y and watch validation accuracy jump.

**Explain:** why do trees exploit leakage aggressively?

**Explain:** why do trees exploit leakage aggressively?

In [ ]:
# Leakage demo
Xtr_leak = np.hstack([Xtr, ytr.reshape(-1,1)])
Xva_leak = np.hstack([Xva, yva.reshape(-1,1)])
clf = DecisionTreeClassifier(max_depth=3, random_state=0)
clf.fit(Xtr_leak, ytr)
print('val acc with leakage', clf.score(Xva_leak, yva))

# Rationale: trees can memorize leakage easily by splitting directly on the leaky feature.

**Why this works.** trees can memorize leakage easily by splitting directly on the leaky feature.